# Capa Bronze – EDA e ingesta cruda · Consumo Alumbrado Público Medellín (Septiembre 2026)

Análisis exploratorio del archivo `Consumo_Sep_2026_Reduccion_Horaria.xlsx` (consumo de energía y valor en pesos del alumbrado público de Medellín) y aterrizaje **tal cual** (raw, sin transformar) de sus celdas en tablas de la capa Bronze `workspace.fenomeno_nino_bronze`.

In [0]:
%pip install openpyxl

## 1. Configuración y carga cruda del archivo

Se lee el Excel **sin encabezado** (`header=None`): el archivo es un reporte con celdas de formato (títulos, parámetros, secciones y notas), no una tabla plana, por lo que en Bronze se preserva la grilla exacta.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

RUTA_ARCHIVO = "/Workspace/Users/alba.preciado2860@unaula.edu.co/fenomenoNi-o/Consumo_Sep_2026_Reduccion_Horaria.xlsx"
CATALOGO = "workspace"
SCHEMA_BRONZE = "fenomeno_nino_bronze"
NOMBRE_ARCHIVO = "Consumo_Sep_2026_Reduccion_Horaria.xlsx"

xls = pd.ExcelFile(RUTA_ARCHIVO)
print("Hojas del archivo:", xls.sheet_names)

hojas = {hoja: pd.read_excel(RUTA_ARCHIVO, sheet_name=hoja, header=None) for hoja in xls.sheet_names}
for hoja, df in hojas.items():
    print(f"  - '{hoja}': {df.shape[0]} filas × {df.shape[1]} columnas")

## 2. EDA · Estructura del archivo: secciones y filas con contenido

Mapa fila por fila de lo que realmente contiene cada hoja: títulos, parámetros, encabezados de tabla, datos, subtotales/totales y notas.

In [0]:
for hoja, df in hojas.items():
    print(f"\n{'='*90}\nHOJA '{hoja}' — filas con contenido\n{'='*90}")
    for i in range(df.shape[0]):
        fila = df.iloc[i]
        vals = [(j, v) for j, v in enumerate(fila) if pd.notna(v)]
        if vals:
            resumen = " | ".join(f"[c{j+1}] {str(v)[:70]}" for j, v in vals[:7])
            print(f"Fila {i+1:2}: {resumen}" + (" …" if len(vals) > 7 else ""))

## 3. EDA · Perfil de calidad: celdas vacías, tipos de dato y completitud

Cuantifica **dónde** están los vacíos y qué proporción del archivo es numérica vs. texto — insumo para decidir la imputación en Silver.

In [0]:
perfil = []
for hoja, df in hojas.items():
    plano = [v for v in df.values.flatten() if pd.notna(v)]
    n_num = sum(isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) for v in plano)
    perfil.append({
        "hoja": hoja,
        "celdas_totales": df.size,
        "celdas_con_valor": len(plano),
        "celdas_vacias": df.size - len(plano),
        "pct_vacias": round(100 * (df.size - len(plano)) / df.size, 1),
        "valores_numericos": n_num,
        "valores_texto": len(plano) - n_num,
    })
display(pd.DataFrame(perfil))

for hoja, df in hojas.items():
    print(f"\nCeldas vacías por columna — '{hoja}':")
    vacias_col = df.isna().sum()
    for j, n in vacias_col[vacias_col > 0].items():
        print(f"  Columna {j+1}: {n} vacías de {df.shape[0]}")

## 4. EDA · Grillas crudas visualizadas

Visualización completa de cada hoja tal como llega del archivo de origen.

In [0]:
for hoja, df in hojas.items():
    print(f"\nGrilla cruda — '{hoja}':")
    display(df)

## 5. Ingesta a la capa Bronze (Unity Catalog)

Dos representaciones complementarias, ambas inmutables y sin transformar:

* `consumo_raw_celdas` — formato largo: una fila por celda (hoja, fila, columna, valor, tipo), incluidas las vacías. Ideal para trazabilidad e imputación.
* `septiembre_2026_raw` / `reduccion_horaria_raw` — grilla por hoja (texto plano) conservando la forma del Excel.

In [0]:
def col_letra(j):
    """Índice 0-based → letra de columna Excel (0→A, 26→AA)."""
    s, j = "", j + 1
    while j:
        j, r = divmod(j - 1, 26)
        s = chr(65 + r) + s
    return s

registros = []
for hoja, df in hojas.items():
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = df.iat[i, j]
            es_vacio = pd.isna(v)
            es_num = (not es_vacio) and isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)
            registros.append({
                "hoja": hoja,
                "fila_excel": i + 1,
                "col_excel": col_letra(j),
                "valor": None if es_vacio else str(v),
                "tipo_valor": "vacio" if es_vacio else ("numero" if es_num else "texto"),
            })

df_celdas = (spark.createDataFrame(pd.DataFrame(registros))
    .withColumn("archivo_origen", F.lit(NOMBRE_ARCHIVO))
    .withColumn("fecha_ingesta", F.current_timestamp()))
df_celdas.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{SCHEMA_BRONZE}.consumo_raw_celdas")
print(f"consumo_raw_celdas → {df_celdas.count()} filas")

TABLAS_GRID = {"Septiembre 2026": "septiembre_2026_raw", "Reducción horaria": "reduccion_horaria_raw"}
for hoja, df in hojas.items():
    nombre = TABLAS_GRID[hoja]
    g = df.map(lambda v: "" if pd.isna(v) else str(v))
    g.columns = [f"c{j+1}" for j in range(g.shape[1])]
    g.insert(0, "fila_excel", range(1, len(g) + 1))
    df_g = (spark.createDataFrame(g)
        .withColumn("archivo_origen", F.lit(NOMBRE_ARCHIVO))
        .withColumn("fecha_ingesta", F.current_timestamp()))
    df_g.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{SCHEMA_BRONZE}.{nombre}")
    print(f"{nombre} → {df_g.count()} filas × {g.shape[1] - 1} columnas de datos")

## 6. Validación de la ingesta

Verifica que las tablas Bronze existan con el número esperado de filas y que la distribución de tipos de celda sea coherente con lo observado en el EDA.

In [0]:
esperado = {"consumo_raw_celdas": 32*9 + 46*11, "septiembre_2026_raw": 32, "reduccion_horaria_raw": 46}
for tabla, n_esperado in esperado.items():
    n = spark.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{tabla}").count()
    estado = "OK" if n == n_esperado else "REVISAR"
    print(f"{estado} · {CATALOGO}.{SCHEMA_BRONZE}.{tabla}: {n} filas (esperadas {n_esperado})")

display(spark.sql(f"""
  SELECT tipo_valor, COUNT(*) AS n_celdas
  FROM {CATALOGO}.{SCHEMA_BRONZE}.consumo_raw_celdas
  GROUP BY tipo_valor
  ORDER BY n_celdas DESC
"""))